**please run this notebook on google colab**

## downloading and extracting the d3dfacs dataset

In [1]:
#downloading dataset
!wget https://files.is.tue.mpg.de/tbolkart/FLAME/d3dfacs_alignments.zip -P /content/data
!unzip /content/data/d3dfacs_alignments.zip

Streaming output truncated to the last 5000 lines.
  inflating: Michaela/17+24/17+24_011.ply  
  inflating: Michaela/17+24/17+24_012.ply  
  inflating: Michaela/17+24/17+24_013.ply  
  inflating: Michaela/17+24/17+24_014.ply  
  inflating: Michaela/17+24/17+24_015.ply  
  inflating: Michaela/17+24/17+24_016.ply  
  inflating: Michaela/17+24/17+24_017.ply  
  inflating: Michaela/17+24/17+24_018.ply  
  inflating: Michaela/17+24/17+24_019.ply  
  inflating: Michaela/17+24/17+24_020.ply  
  inflating: Michaela/17+24/17+24_021.ply  
  inflating: Michaela/17+24/17+24_022.ply  
  inflating: Michaela/17+24/17+24_023.ply  
  inflating: Michaela/17+24/17+24_024.ply  
  inflating: Michaela/17+24/17+24_025.ply  
  inflating: Michaela/17+24/17+24_026.ply  
  inflating: Michaela/17+24/17+24_027.ply  
  inflating: Michaela/17+24/17+24_028.ply  
  inflating: Michaela/17+24/17+24_029.ply  
  inflating: Michaela/17+24/17+24_030.ply  
  inflating: Michaela/17+24/17+24_031.ply  
  inflating: Michaela/17+

## install open3d

In [2]:
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.0 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfully uninstalled Werkzeug-3.1.3
  Attempting uninstall: flask
    Found existing installation: Flask 3.1.0
    Uninst

## imports and usefull functions

In [3]:
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

In [22]:
def draw_geometries(geometries, points_to_draw):
    graph_objects = []

    for geometry in geometries:

        points = np.asarray(geometry.points)

        colors = None
        if geometry.has_colors():
            colors = np.asarray(geometry.colors)
        elif geometry.has_normals():
            colors = (0.5, 0.5, 0.5) + np.asarray(geometry.normals) * 0.5
        else:
            geometry.paint_uniform_color((1.0, 0.0, 0.0))
            colors = np.asarray(geometry.colors)

        scatter_3d = go.Scatter3d(x=points[:,0], y=points[:,1], z=points[:,2], mode='markers', marker=dict(size=1, color=colors))
        graph_objects.append(scatter_3d)
        for point in points_to_draw:
            nose_tip = point
            nose_tip_3d = go.Scatter3d(x=[nose_tip[0]], y=[nose_tip[1]], z=[nose_tip[2]], mode='markers', marker=dict(size=2, color='blue'))
            graph_objects.append(nose_tip_3d)

    fig = go.Figure(
        data=graph_objects,
        layout=dict(
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False)
            )
        )
    )

    fig.show()


def find_middle_point(cluster):
    """ find the X-axis middle point of a cluster

    Args:
        cluster (np.array): cluster

    Returns:
        int: middle point X-axis
    """

    min_x = np.min(cluster[:, 0])
    max_x = np.max(cluster[:, 0])

    return (min_x + max_x) / 2


o3d.visualization.draw_geometries = draw_geometries

## first solution

reading a sample cloud file and ploting it with the center

In [11]:
input_file = "/content/Gwenda/1+2/1+2_065.ply"
# input_file = "Data/someone2.ply"
cloud = o3d.io.read_point_cloud(input_file) # Read the point cloud
o3d.visualization.draw_geometries([cloud], points_to_draw=[[0,0,0]])

points = np.asarray(cloud.points)
points.shape

(5023, 3)

drawing points with min and max values in each dim

In [12]:
draw_idx = np.concatenate((np.argmax(points, axis=0), np.argmin(points, axis=0)))
o3d.visualization.draw_geometries([cloud], points_to_draw=[points[x] for x in draw_idx])

drawing most protruding point

In [13]:
nose_tip = points[draw_idx[2]]
o3d.visualization.draw_geometries([cloud], points_to_draw=[nose_tip])

onther sample

In [15]:
input_file = "/content/Joe/1+2/1+2_175.ply"
cloud = o3d.io.read_point_cloud(input_file) # Read the point cloud

points = np.asarray(cloud.points)

draw_idx = np.concatenate((np.argmax(points, axis=0), np.argmin(points, axis=0)))

nose_tip = points[draw_idx[2]]
o3d.visualization.draw_geometries([cloud], points_to_draw=[nose_tip])

as we can see, first solution does not work for this sample

## second solution

performing DBSCAN clusrintg

In [20]:
# Perform DBSCAN clustering
dbscan = DBSCAN(eps=0.009, min_samples=24) # Adjust eps and min_samples as needed
labels = dbscan.fit_predict(points)

# Create a new point cloud with colors based on cluster labels
colored_cloud = o3d.geometry.PointCloud()
colored_cloud.points = o3d.utility.Vector3dVector(points)

# Assign colors to clusters
colors = plt.cm.get_cmap("tab20", np.max(labels) + 2)(labels)
colors[labels < 0] = 0 #Assign black color to outliers
colored_cloud.colors = o3d.utility.Vector3dVector(colors[:, :3])

# Visualize the clustered point cloud
o3d.visualization.draw_geometries([colored_cloud], points_to_draw=[])

<ipython-input-20-77a79e79a919>:10: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.



as we can see, DBSCAN can cluter the eyes, ears, nose and the lips

slecting the cluster with the second min y and finding points in this cluster
with min and max values in each dim

In [21]:
# Find the cluster with the second minimum y-coordinate
unique_labels = np.unique(labels)
unique_labels = unique_labels[unique_labels != -1]  # Exclude noise points

min_y_values = []
for cluster_label in unique_labels:
    cluster_points = points[labels == cluster_label]
    min_y_values.append(np.min(cluster_points[:, 1]))

sorted_indices = np.argsort(min_y_values)
second_min_y_cluster_index = sorted_indices[1]
second_min_y_cluster_label = unique_labels[second_min_y_cluster_index]


print(f"Cluster with the second minimum y-coordinate: {second_min_y_cluster_label}")

# Visualize the cluster with the second minimum y-coordinate
second_min_y_cluster_points = points[labels == second_min_y_cluster_label]



# Find min and max points in each dimension for the second minimum y cluster
min_max_indices = []
for dim in range(3):  # Iterate over x, y, and z dimensions
    min_idx = np.argmin(second_min_y_cluster_points[:, dim])
    max_idx = np.argmax(second_min_y_cluster_points[:, dim])
    min_max_indices.extend([min_idx, max_idx])

# Visualize the cluster with the second minimum y-coordinate and highlighted min/max points
min_max_points = [second_min_y_cluster_points[i] for i in min_max_indices]
o3d.visualization.draw_geometries([colored_cloud], points_to_draw = min_max_points)


Cluster with the second minimum y-coordinate: 2


now we will select most protruding point in nose cluster. for y and z we will use this points. for x axis we will consider the avg x between min and max x values in this cluster

In [23]:
nose_cluster = second_min_y_cluster_points
nose_tip_x = find_middle_point(second_min_y_cluster_points)
most_protruding_point = nose_cluster[np.argmax(nose_cluster[:, 2])]
nose_tip = np.array([nose_tip_x, most_protruding_point[1], most_protruding_point[2]])

In [24]:
o3d.visualization.draw_geometries([cloud], points_to_draw = [nose_tip])